In [ ]:
import numpy as np
from pyscf import gto, dft, lo, ao2mo, fci
import matplotlib.pyplot as plt


class DIIS:
    def __init__(self, max_vecs=6):
        self.max_vecs   = max_vecs
        self.vecs       = []   
        self.errors     = []  
        self._err_shape = None

    def update(self, u_vec, err):
        self._err_shape = np.asarray(err).shape
        self.errors.append(np.asarray(err).ravel().copy())
        self.vecs.append(np.asarray(u_vec).ravel().copy())

        if len(self.vecs) > self.max_vecs:
            self.errors.pop(0)
            self.vecs.pop(0)

    def extrapolate(self):
        n = len(self.errors)
        if n == 0:
            raise RuntimeError("No vectors stored in DIIS.")

        E = np.array(self.errors)      

        B = np.zeros((n + 1, n + 1))
        B[:-1, :-1] = E @ E.T             
        B[-1, :-1]  = -1
        B[:-1, -1]  = -1

        rhs = np.zeros(n + 1)
        rhs[-1] = -1

        try:
            coeffs = np.linalg.solve(B, rhs)
        except np.linalg.LinAlgError:
            
            return np.array(self.vecs[-1])

        u_extrap = coeffs[:n] @ np.array(self.vecs)
        return u_extrap                                

xyz = """
H   0.000   0.000   0.000
H   0.000   0.000   1.800
H   0.000   0.000   3.600
H   0.000   0.000   5.400
H   0.000   0.000   7.200
H   0.000   0.000   9.000
H   0.000   0.000  10.800
H   0.000   0.000  12.600
"""

mol = gto.Mole()
mol.atom  = xyz
mol.basis = "sto-3g"
mol.build()

mf = dft.RKS(mol, xc="PBE0").run()
print("SCF Energy:", mf.e_tot)

# Localization (occupied only)

n_occ    = mol.nelectron // 2
loc_coeff = lo.PM(mol).kernel(mf.mo_coeff)

S         = mol.intor("int1e_ovlp")
ao_labels = mol.ao_labels(fmt=False)
natm      = mol.natm

ao_by_atom = [[] for _ in range(natm)]
for idx, (aidx, *_) in enumerate(ao_labels):
    ao_by_atom[aidx].append(idx)

def per_orbital_mulliken(c_vec, occ_num=2.0):
    c_vec = np.asarray(c_vec).ravel()
    DSc   = occ_num * (c_vec @ S) * c_vec
    return np.array([np.sum(DSc[idxs]) for idxs in ao_by_atom])

# Fragment Selection

FRAG_ATOMS = [2]
HIGH_THRSH = 0.30

fragment_orbs  = []
environment_orbs = []
frag_scores    = []

for i in range(n_occ):
    pops      = per_orbital_mulliken(loc_coeff[:, i], occ_num=2.0)
    frag_part = pops[FRAG_ATOMS].sum() / 2.0
    frag_scores.append((i, frag_part))

for i, frag_part in frag_scores:
    if frag_part > HIGH_THRSH:
        fragment_orbs.append(i)
    else:
        environment_orbs.append(i)

if len(fragment_orbs) == 0:
    print("Warning: No orbitals passed threshold. Using fallback selection.")
    frag_scores.sort(key=lambda x: x[1], reverse=True)
    fragment_orbs    = [frag_scores[0][0]]
    environment_orbs = [i for i, _ in frag_scores[1:]]

print("Fragment orbitals :", fragment_orbs)
print("Environment orbitals:", environment_orbs)

frag_idx = np.array(fragment_orbs)
env_idx  = np.array(environment_orbs)

# Initial density

D_ao = mf.make_rdm1()
D_mo = loc_coeff.T @ S @ D_ao @ S @ loc_coeff

# DMET parameters

MAX_DMET_ITER = 30
D_TOL         = 1.2e-3
ALPHA         = 0.4   
MIX           = 0.5   

u = np.zeros(len(fragment_orbs))
orig_get_hcore = mf.get_hcore

def make_hcore_func(base_hcore_fn, coeff_frag, u_vec, S_mat):
    def get_hcore_with_u(*args):
        hcore = base_hcore_fn(*args).copy()
        for i in range(len(u_vec)):
            vec  = coeff_frag[:, i]
            proj = np.outer(vec, vec) @ S_mat
            hcore -= u_vec[i] * proj
        return hcore
    return get_hcore_with_u

# DMET loop

densities = []
diis      = DIIS(max_vecs=6)

for dmet_iter in range(MAX_DMET_ITER):

    print(f"\n--- DMET Iteration {dmet_iter} ---")

    frag_block = loc_coeff[:, fragment_orbs]
    hcore_fn   = make_hcore_func(orig_get_hcore, frag_block, u, S)

    mf_dmet           = dft.RKS(mol, xc="PBE0")
    mf_dmet.get_hcore = hcore_fn
    mf_dmet.conv_tol  = 1e-9
    mf_dmet.max_cycle = 300

    dm0 = D_ao if dmet_iter > 0 else None
    mf_dmet.kernel(dm0=dm0)

    D_ao = mf_dmet.make_rdm1()
    D_mo = loc_coeff.T @ S @ D_ao @ S @ loc_coeff

    print("MF fragment occ:", np.trace(D_mo[np.ix_(frag_idx, frag_idx)]))

    # Bath construction

    D_EF              = D_mo[np.ix_(env_idx, frag_idx)]  
    U_svd, svals, _   = np.linalg.svd(D_EF, full_matrices=False)

    MAX_BATH = len(fragment_orbs)

    nbath = min(MAX_BATH, len(env_idx), len(svals))

    if nbath == 0:
        raise ValueError("No bath orbitals found — environment is empty.")
    U_bath  = U_svd[:, :nbath]                       
    print(f"Bath singular values: {svals[:nbath]}")

    bath_ao = loc_coeff[:, env_idx] @ U_bath     

    C_emb = np.hstack((loc_coeff[:, fragment_orbs], bath_ao))
    n_emb = C_emb.shape[1]
    print(f"Embedded space: {n_emb} orbitals ({len(fragment_orbs)} frag + {nbath} bath)")

    # Embedded Hamiltonian

    hcore_ao = orig_get_hcore().real
    h1  = C_emb.T @ hcore_ao @ C_emb
    eri = ao2mo.kernel(mol, C_emb, compact=False).reshape(n_emb, n_emb, n_emb, n_emb)

    # Electron count (via projected density)

    N_frag           = np.trace(D_mo[np.ix_(frag_idx, frag_idx)])
    N_bath           = np.trace(U_bath.T @ D_mo[np.ix_(env_idx, env_idx)] @ U_bath)
    N_emb_electrons  = int(round(N_frag + N_bath))
    if N_emb_electrons % 2 != 0:
        N_emb_electrons += 1
    nelec = (N_emb_electrons // 2, N_emb_electrons // 2)
    print(f"Electrons in impurity: {N_emb_electrons}  (frag {N_frag:.3f}, bath {N_bath:.3f})")

    # FCI solver

    cisolver          = fci.direct_spin1.FCI()
    cisolver.conv_tol = 1e-10
    E_imp, fcivec     = cisolver.kernel(h1, eri, n_emb, nelec)
    print(f"Impurity energy: {E_imp:.8f}")

    # Density matrices

    D_emb      = cisolver.make_rdm1(fcivec, n_emb, nelec)
    nf         = len(fragment_orbs)
    D_frag_imp = D_emb[:nf, :nf]
    D_frag_mf  = D_mo[np.ix_(frag_idx, frag_idx)]

    density_error = np.linalg.norm(D_frag_imp - D_frag_mf)
    densities.append(density_error)
    print(f"Density error: {density_error:.2e}")

    if density_error < D_TOL:
        print("DMET Converged!")
        break


    diag_imp = np.diag(D_frag_imp)
    diag_mf  = np.diag(D_frag_mf)
    diff     = diag_imp - diag_mf     

    u_new = u + ALPHA * diff

    u_new = MIX * u_new + (1 - MIX) * u

    diis.update(u_new, diff)
    if dmet_iter >= 2:
        u_new = diis.extrapolate()    

    u = np.clip(u_new, -5.0, 5.0)
    print(f"Correlation potential u: {u}")

# Final energy

E_dmet = E_imp - np.dot(u, np.diag(D_frag_imp))
print("\n========== Final Results ==========")
print(f"DMET Energy:  {E_dmet:.8f}")
print(f"Total iters:  {len(densities)}")

# Plot convergence

plt.figure(figsize=(7, 4))
plt.plot(range(len(densities)), densities, marker="o")
plt.xlabel("DMET Iteration")
plt.ylabel("Density Error ||ΔD||")
plt.title("DMET Convergence")
plt.yscale("log")
plt.axhline(D_TOL, ls="--", color="gray", label=f"tol = {D_TOL}")
plt.legend()
plt.tight_layout()
plt.savefig("dmet_convergence.png", dpi=150)
plt.show()

converged SCF energy = -3.74224731339111
SCF Energy: -3.742247313391111
Fragment orbitals : [2]
Environment orbitals: [3, 0, 1]

--- DMET Iteration 0 ---


Overwritten attributes  get_hcore  of <class 'pyscf.dft.rks.RKS'>


converged SCF energy = -3.74224731339111
MF fragment occ: 0.8204843031594747
Bath singular values: [0.35410625]
Embedded space: 2 orbitals (1 frag + 1 bath)
Electrons in impurity: 2  (frag 0.820, bath 1.480)
Impurity energy: -2.80040311
Density error: 7.66e-01
Correlation potential u: [-0.15327638]

--- DMET Iteration 1 ---
